# 02 - Which regressor best predicts RT? (paper Table 2)

The paper compares 8 candidate regressors for RT (product, sum, sum^2, and
the log/sqrt versions of those, plus the min operand) via AIC, separately
for tie and non-tie problems in each table. Lower AIC = better fit.

Uses a random-intercept-only mixed model per regressor (the paper uses
random intercept *and* slope per regressor, which is far more expensive to
fit 8x2x2 times -- this is a simplification worth revisiting once real data
volume makes the richer model affordable).


In [1]:
# Data source toggle. The live backend DB has zero trial_results as of
# 2026-09 (see ../DATA_GAPS.md) -- this app just went public. Flip
# USE_SYNTHETIC to False once real rows exist; every cell below is written
# against the DataFrame shape returned by moravec_analysis.data, so nothing
# else needs to change.
USE_SYNTHETIC = True

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from moravec_analysis.data import load_trial_results
from moravec_analysis.features import add_all_derived_fields, filter_rt_outliers
from moravec_analysis.synth import generate_synthetic_trials

if USE_SYNTHETIC:
    raw = generate_synthetic_trials(n_users=400, trials_per_user=150, seed=7)
else:
    raw = load_trial_results()  # defaults to apps/backend/data/moravec.sqlite

df = add_all_derived_fields(raw)
print(f"{len(df):,} trials, {df.email_hash.nunique():,} users")
df.head()


61,859 trials, 400 users


,id,email_hash,level_number,category_codename,operands,answer,correct,time_exceeded,time_taken,played_at,...,has_five,op1_gt_op2,unordered_pair,presented_pair,is_rhyme_pair,presented_in_rhyme_order,is_rhyme_control_pair,presented_in_control_order,table_distance_1,numeric_distance_le_2
0,25021f60-b6de-46ea-a1a9-c4478352ca5c,f6bd0410527d6ea9a1e60637a73f97679f49abea307385...,5,1d+1d,"[2, 2]",4,True,False,1783.132304,2026-08-26 16:06:24.275,...,False,False,"(2, 2)","(2, 2)",False,False,False,False,<NA>,<NA>
1,7f142f53-5ae5-4925-af45-07e9777b1356,f6bd0410527d6ea9a1e60637a73f97679f49abea307385...,5,1dx1d,"[3, 7]",21,True,False,3248.066102,2026-08-26 10:19:56.252,...,False,False,"(3, 7)","(3, 7)",False,False,False,False,<NA>,<NA>
2,51d7e69b-5cd8-47dd-934a-44883b9cb443,f6bd0410527d6ea9a1e60637a73f97679f49abea307385...,4,1d+1d,"[8, 0]",8,True,False,1584.694315,2026-08-21 05:32:44.299,...,False,True,"(0, 8)","(8, 0)",False,False,False,False,<NA>,<NA>
3,d3e77146-5ce5-40fa-b9d0-8601e9958f1b,f6bd0410527d6ea9a1e60637a73f97679f49abea307385...,1,1d+1d,"[4, 4]",8,True,False,1936.467264,2026-08-15 14:45:52.252,...,False,False,"(4, 4)","(4, 4)",False,False,False,False,<NA>,<NA>
4,2a41aec6-43c7-4aba-930a-274b285688de,f6bd0410527d6ea9a1e60637a73f97679f49abea307385...,5,1dx1d,"[5, 5]",25,True,False,1939.174017,2026-08-18 17:28:00.207,...,True,False,"(5, 5)","(5, 5)",False,False,False,False,<NA>,<NA>


In [2]:
one_by_one = df[df["category_codename"].isin(["1d+1d", "1dx1d"]) & df["correct"]]
clean = filter_rt_outliers(one_by_one, group_col="category_codename", sd=4.0)

REGRESSORS = [
    "product", "sum", "sum_sq",
    "log_product", "log_sum", "log_sum_sq",
    "sqrt_product", "sqrt_sum", "min_operand",
]


In [3]:
import statsmodels.formula.api as smf

def aic_table(data):
    rows = []
    for category, cat_label in [("1d+1d", "ADDITION"), ("1dx1d", "MULTIPLICATION")]:
        for is_tie, tie_label in [(False, "nonties"), (True, "ties")]:
            sub = data[(data["category_codename"] == category) & (data["is_tie"] == is_tie)]
            row = {"category": cat_label, "group": tie_label}
            for reg in REGRESSORS:
                model = smf.mixedlm(f"time_taken ~ {reg}", data=sub, groups=sub["email_hash"])
                row[reg] = model.fit(reml=False).aic
            rows.append(row)
    return pd.DataFrame(rows).set_index(["category", "group"])


table2 = aic_table(clean)
table2.style.highlight_min(axis=1, color="#a8dadc")


/home/juancuiule/eglc-moravec/packages/analysis/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  row[reg] = model.fit(reml=False).aic


/home/juancuiule/eglc-moravec/packages/analysis/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  row[reg] = model.fit(reml=False).aic


/home/juancuiule/eglc-moravec/packages/analysis/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  row[reg] = model.fit(reml=False).aic


/home/juancuiule/eglc-moravec/packages/analysis/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  row[reg] = model.fit(reml=False).aic


/home/juancuiule/eglc-moravec/packages/analysis/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2384: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  rslt = super().fit(
/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: Retrying MixedLM optimization with cg
  row[reg] = model.fit(reml=False).aic


...

/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  row[reg] = model.fit(reml=False).aic


/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  row[reg] = model.fit(reml=False).aic


/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  row[reg] = model.fit(reml=False).aic


/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  row[reg] = model.fit(reml=False).aic


/tmp/ipykernel_468435/4274414805.py:11: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  row[reg] = model.fit(reml=False).aic
